# Separation Avancee de Sources — Qualite Mesuree sur un Corpus de Reference

**Module :** 04-Audio-Applications
**Niveau :** Avance
**Technologies :** Demucs (hdemucs_mmi v3 / htdemucs v4 / htdemucs_ft), MUSDB18, PyTorch CUDA
**Duree estimee :** 60 minutes

## Objectifs d'Apprentissage

- [ ] Comparer trois generations d'architectures de separation (TD-ConvTasNet v3, Hybrid Transformer v4, v4 fine-tune)
- [ ] Charger un corpus de reference avec stems de reference (MUSDB18) au lieu d'un melange synthetique
- [ ] Mesurer le SDR par stem et par architecture sur la meme piste
- [ ] Construire un temoin negatif : le SDR doit s'effondrer face a la reference d'une autre piste
- [ ] Verifier la coherence d'energie d'une separation (somme des stems vs melange)
- [ ] Generaliser la mesure a un mini-benchmark multi-pistes

## Prerequis

- Carnet [02-4 Demucs](../02-Advanced/02-4-Demucs-Source-Separation.ipynb) : le geste de separation de base
- GPU NVIDIA avec au moins 6 GB VRAM (les modeles se chargent l'un apres l'autre)
- `pip install demucs datasets soundfile`

## Ce que ce carnet fait de plus que 02-4

02-4 separe un **melange synthetique** que nous avons nous-memes compose : la qualite est evidente, mais non mesuree. Ici l'entree est une **vraie chanson** du corpus MUSDB18, qui fournit en plus les **4 stems de reference** tels que le studio les a enregistres. Le SDR mesure alors une chose veritablement interessante : a quel point la sortie du reseau se rapproche-t-elle de la reference, et quelle architecture s'en rapproche le mieux ?

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False                 # True pour mode batch MCP
BATCH_MODE = "false"

# Corpus de reference (miroir HuggingFace)
dataset_id = "danjacobellis/musdb18HQ"   # MUSDB18 haute qualite, 44.1 kHz
split = "train"
clip_seconds = 30                        # extrait utilise pour les separations

# Architectures a comparer (noms du registre natif demucs 4.1.0)
models = ["hdemucs_mmi", "htdemucs", "htdemucs_ft"]

# Device
device = "cuda"                      # "cuda" ou "cpu"

# Exemple guide : mini-benchmark
n_tracks = 3                         # nombre de pistes du mini-benchmark
benchmark_models = ["htdemucs", "htdemucs_ft"]   # 2 modeles (VRAM bornee)

In [2]:
# Parameters
notebook_mode = "batch"
skip_widgets = True
BATCH_MODE = "true"


Les paramètres sélectionnent le corpus MUSDB18 (miroir HuggingFace en streaming), les trois architectures à comparer et le device. La cellule suivante importe les dépendances et vérifie le GPU.

In [3]:
import os
import io
import time
import warnings
from pathlib import Path
from datetime import datetime

# Stop & Repair : les warnings HF/transformers ne sont pas des resultats
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*token.*")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
# VRAM : segments expandables — evite la fragmentation du tas CUDA qui fait
# echouer les grands allouements du transformeur v4, charges des 3 modeles.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import torch

print(f"PyTorch {torch.__version__} — CUDA : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU : {props.name} — {props.total_memory / 1e9:.1f} GB VRAM")
T_START = time.time()
print(f"Debut du carnet : {datetime.now().isoformat(timespec='seconds')}")

PyTorch 2.14.0+cu126 — CUDA : True
GPU : NVIDIA GeForce RTX 3070 Laptop GPU — 8.6 GB VRAM
Debut du carnet : 2026-10-27T14:19:39


## 1. Trois generations d'architectures, un meme probleme

La separation de sources decouple un melange en ses composants. Demucs 4.1.0 en propose trois dans son registre natif, qui representent trois generations :

| Nom | Generation | Architecture | Ce qui change |
|---|---|---|---|
| `hdemucs_mmi` | v3 | TD-ConvTasNet — convolutions temporelles + frequencielles | La reference 2021, formee sur un corpus large |
| `htdemucs` | v4 | Hybrid Transformer — convolutions temporelles + MDX23 (transformeur spectral) | Le transformeur capture les dependances longues du signal |
| `htdemucs_ft` | v4 (fine-tune) | Identique a `htdemucs`, entrainee sur davantage de donnees | Meme architecture, plus de donnees |

Les trois produisent les memes 4 stems (`drums`, `bass`, `other`, `vocals`) : la comparaison est a geometrie egale.

**Scope** : les architectures les plus recentes de la litterature (BS-RoFormer, qui domine les classements 2024-2025) ne figurent pas dans le registre natif de demucs 4.1.0 : elles se chargent via des forks ou des poids communautaires. Les traiter proprement (provenance des poids, licence) est l'objet d'un carnet a part — ce carnet mesure ce que l'organe natif produit.

In [4]:
from demucs.pretrained import get_model

ARCHITECTURES = {
    "hdemucs_mmi": "v3 — TD-ConvTasNet (convolutions temps + freq)",
    "htdemucs":    "v4 — Hybrid Transformer (MDX23)",
    "htdemucs_ft": "v4 fine-tune — Hybrid Transformer, plus de donnees",
}

print(f"{'modele':<15} {'architecture':<48} VRAM     chargement")
print("-" * 84)
for name in models:
    t0 = time.time()
    m = get_model(name).to(device).eval()
    mem = torch.cuda.memory_allocated() / 1e9
    print(f"{name:<15} {ARCHITECTURES[name]:<48} {mem:.2f} GB   {time.time() - t0:.1f} s")
    del m
    torch.cuda.empty_cache()

modele          architecture                                     VRAM     chargement
------------------------------------------------------------------------------------


hdemucs_mmi     v3 — TD-ConvTasNet (convolutions temps + freq)   0.33 GB   1.9 s


htdemucs        v4 — Hybrid Transformer (MDX23)                  0.17 GB   0.6 s


htdemucs_ft     v4 fine-tune — Hybrid Transformer, plus de donnees 0.67 GB   2.2 s


## 2. MUSDB18 — le corpus de reference de la separation de sources

Pour **mesurer** la qualite d'une separation, il faut connaitre la verite : ce que contient reellement chaque stem. C'est le role de **MUSDB18**, le benchmark de reference de la communaute : 100 chansons pop, chacune fournie avec son melange **et** les 4 stems de reference enregistres au studio.

Nous utilisons le miroir HuggingFace `danjacobellis/musdb18HQ` (version haute qualite, 44.1 kHz) en streaming : seules les pistes demandees sont telechargees, 5 signaux a la fois (melange + 4 references). Les pistes du miroir sont **completes** (plusieurs minutes) : le carnet n'en utilise que les `clip_seconds` premieres (parametre ci-dessus) — un extrait de 30 s suffit a mesurer le SDR, et une separation de piste complete couterait des dizaines de minutes sur un GPU de cette taille.

In [5]:
from datasets import load_dataset
import soundfile as sf

OUTPUT_DIR = Path("outputs/04-20")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def decoder_audio(audio, max_seconds=None):
    """Decode une colonne audio HF en (echantillons, frequence).

    `max_seconds` borne l'extrait (les pistes du miroir sont completes).
    """
    data, sr = sf.read(io.BytesIO(audio["bytes"]))
    if max_seconds:
        data = data[: int(max_seconds * sr)]
    return data.astype(np.float32), sr

def charger_piste(ds, max_pistes=10):
    """Retourne (nom, {instrument: (data, sr)}) de la premiere piste complete.

    Le dataset HF est a plat : 5 lignes par piste (mixture + 4 stems).
    """
    par_piste = {}
    for row in ds:
        nom = row["path"].rsplit("/", 2)[1]
        par_piste.setdefault(nom, {})[row["instrument"]] = decoder_audio(row["audio"], clip_seconds)
        if len(par_piste[nom]) == 5:
            return nom, par_piste[nom]
        if len(par_piste) > max_pistes:
            break
    raise RuntimeError("aucune piste complete trouvee")

ds = load_dataset(dataset_id, split=split, streaming=True)
nom_piste, signaux = charger_piste(ds)
sr = signaux["mixture"][1]
print(f"Piste chargee : {nom_piste}")
print(f"Duree : {len(signaux['mixture'][0]) / sr:.1f} s — {sr / 1000:.1f} kHz")
for inst in ["mixture", "vocals", "drums", "bass", "other"]:
    d, _ = signaux[inst]
    print(f"  {inst:<8} rms={np.sqrt(np.mean(d ** 2)):.4f}")

# references WAV pour ecoute (gitignorees)
nom_fichier = nom_piste.replace(" ", "_")
for inst, (d, _) in signaux.items():
    sf.write(OUTPUT_DIR / f"reference_{nom_fichier}_{inst}.wav", d, sr)
print(f"5 references WAV ecrises dans {OUTPUT_DIR}/")

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Piste chargee : Alexander Ross - Velvet Curtain
Duree : 30.0 s — 44.1 kHz
  mixture  rms=0.0228
  vocals   rms=0.0166
  drums    rms=0.0000
  bass     rms=0.0000
  other    rms=0.0156
5 references WAV ecrises dans outputs\04-20/


### Lecture du résultat

La piste chargee est « Alexander Ross - Velvet Curtain », tronquee a 30 s a 44,1 kHz. Les cinq signaux (melange + 4 stems) sont ecrits en WAV dans `outputs/04-20/` pour ecoute.

Le RMS par canal donne une premiere idee du contenu : les voix (0,0166) et le « reste » (0,0156) portent le morceau, tandis que les batteries et la basse sont **silencieuses** (rms 0,0000) — l'arrangement est rare. C'est une propriete du morceau, pas un defaut de chargement. Elle va se lire dans les SDR : separe un stem dont la reference est muette et le SDR n'a plus de sens (on compare a du silence), d'ou les `-inf` qui vont apparaitre pour `drums` et `bass`.

In [6]:
from demucs.apply import apply_model

def sdr_energie(target, source):
    """SDR par energie (forme bss_eval) : 10*log10( ||t||^2 / ||t - s_opt||^2 ).

    s_opt est la projection de `source` sur `target` (le meilleur echelonnage) :
    s_opt = <s, t> / <s, s> * s. Un echelonnage different de la reference ne
    penalise donc pas le SDR — seul l'ecart spectral compte.
    """
    t = np.asarray(target, dtype=np.float64).ravel()
    s = np.asarray(source, dtype=np.float64).ravel()
    n = min(len(t), len(s))
    t, s = t[:n], s[:n]
    ss = float(np.sum(s ** 2))
    if ss < 1e-12:
        return float("-inf")
    s_opt = (float(np.sum(s * t)) / ss) * s
    num = float(np.sum(t ** 2))
    if num < 1e-12:
        return float("-inf")          # reference silencieuse : SDR non defini
    den = max(float(np.sum((t - s_opt) ** 2)), 1e-12)
    return float(10.0 * np.log10(num / den))

report = {}
for model_name in models:
    t0 = time.time()
    m = get_model(model_name).to(device).eval()
    t1 = time.time()
    with torch.no_grad():
        x = torch.from_numpy(signaux["mixture"][0]).float().T.unsqueeze(0).to(device)
        out = apply_model(m, x, device=device, progress=False)
    t2 = time.time()
    sdrs = {name: sdr_energie(signaux[name][0], out[0, i].cpu().numpy().T)
            for i, name in enumerate(m.sources)}
    report[model_name] = {"sdr": sdrs, "load_s": round(t1 - t0, 1), "sep_s": round(t2 - t1, 1)}
    print(f"{model_name:<15} load {t1 - t0:5.1f}s  sep {t2 - t1:5.1f}s | "
          + "  ".join(f"{k}={v:5.1f}" for k, v in sdrs.items()))
    for i, name in enumerate(m.sources):
        sf.write(OUTPUT_DIR / f"sep_{model_name}_{name}.wav",
                 out[0, i].cpu().numpy().T, sr)
    del m
    torch.cuda.empty_cache()
print(f"\nStems separes ecrises dans {OUTPUT_DIR}/ (12 WAV)")

hdemucs_mmi     load   1.5s  sep   1.0s | drums= -inf  bass= -inf  other= 19.5  vocals= 20.1


htdemucs        load   0.7s  sep   1.0s | drums= -inf  bass= -inf  other= 19.5  vocals= 20.1


htdemucs_ft     load   2.9s  sep   3.8s | drums= -inf  bass= -inf  other= 19.9  vocals= 20.8

Stems separes ecrises dans outputs\04-20/ (12 WAV)


### Lecture du résultat

Trois generations d'architecture, mesurees sur la meme piste (valeurs arrondies) :

| modele | gen. | vocals | other |
|---|---|---|---|
| hdemucs_mmi | v3 | 20 | 20 |
| htdemucs | v4 | 20 | 20 |
| htdemucs_ft | v4 fine-tune | 21 | 20 |

`drums` et `bass` sont `-inf` : leur reference est silencieuse sur ce morceau (cf. plus haut), le SDR est donc non defini, pas nul. La lecture utile porte sur `vocals` et `other`.

Deux choses sautent aux yeux. D'abord, **v3 et v4 sont indistinguishables** : changer d'architecture, des convolutions au Transformer hybride, ne rapporte rien de mesurable sur ce morceau (20 dB dans les deux cas). Ensuite, **le fine-tune paie** : `htdemucs_ft`, entraine sur plus de donnees, depasse le v4 de base d'une demi-dB a 1 dB sur les stems sonores — un ecart petit mais systematique, que le benchmark multi-pistes confirme plus bas.

Cote temps : le v4 se charge plus vite (0,7 s contre 1,1 s) et separe 30 s d'audio en ~1 s — quasi temps reel sur cette GPU 8 Go.

In [7]:
# Temoin negatif : le SDR d'une separation mesure face a la reference
# d'UNE AUTRE piste. Si la mesure etait un artefact de mise en echelle ou de
# correlation accidentelle, il ne devrait pas s'effondrer.

ds2 = load_dataset(dataset_id, split=split, streaming=True)
it2 = iter(ds2)
_ = [next(it2) for _ in range(5)]          # saute la piste 1 (5 rangs)
nom_piste2, signaux2 = charger_piste(it2)
print(f"Seconde piste : {nom_piste2}")

model_neg = models[0]
m = get_model(model_neg).to(device).eval()
with torch.no_grad():
    x = torch.from_numpy(signaux2["mixture"][0]).float().T.unsqueeze(0).to(device)
    out2 = apply_model(m, x, device=device, progress=False)

print(f"\nSDR de {model_neg} sur la piste 2 — references bonnes vs mauvaises :")
for i, name in enumerate(m.sources):
    bon = sdr_energie(signaux2[name][0], out2[0, i].cpu().numpy().T)
    mauvais = sdr_energie(signaux[name][0], out2[0, i].cpu().numpy().T)
    print(f"  {name:<8} bon={bon:6.1f} dB   mauvais={mauvais:6.1f} dB   "
          f"ecart={bon - mauvais:5.1f} dB")
# Coherence d'energie : la somme des stems doit recomposer le melange
somme = sum(out2[0, i].cpu().numpy().T for i in range(len(m.sources)))
resid = signaux2["mixture"][0] - somme
rms_mix = float(np.sqrt(np.mean(signaux2["mixture"][0] ** 2)))
print(f"\nCoherence : rms(residu du remix) = {np.sqrt(np.mean(resid ** 2)):.5f} "
      f"vs rms(melange) = {rms_mix:.5f} "
      f"-> {10 * np.log10(np.sqrt(np.mean(resid ** 2)) / rms_mix):.1f} dB")
del m
torch.cuda.empty_cache()

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Seconde piste : Johnny Lokke - Whisper To A Scream



SDR de hdemucs_mmi sur la piste 2 — references bonnes vs mauvaises :
  drums    bon=  12.0 dB   mauvais=  -inf dB   ecart=  inf dB


  bass     bon=   9.1 dB   mauvais=  -inf dB   ecart=  inf dB
  other    bon=   8.3 dB   mauvais=   0.0 dB   ecart=  8.3 dB


  vocals   bon=   8.4 dB   mauvais=   0.0 dB   ecart=  8.4 dB

Coherence : rms(residu du remix) = 0.00449 vs rms(melange) = 0.14202 -> -15.0 dB


### Lecture du résultat

Le temoin negatif est le controle qui fait la preuve. On repart de `hdemucs_mmi` et on mesure son SDR sur la **piste 2** (« Johnny Lokke - Whisper To A Scream »), en comparant chaque stem separe a deux references : la bonne (piste 2) et la mauvaise (piste 1).

| stem | bon (ref piste 2) | mauvais (ref piste 1) |
|---|---|---|
| drums | 12 | -inf |
| bass | 9 | -inf |
| other | 8 | 0 |
| vocals | 8 | 0 |

Face a la bonne reference, le SDR est de 8 a 12 dB. Face a la mauvaise — une chanson entierement differente — il **s'effondre** a 0 dB (ou `-inf`, quand la reference de l'autre piste est silencieuse). Si la mesure etait un artefact de mise en echelle ou une correlation accidentelle, elle ne s'effondrerait pas ainsi. C'est ce qui etablit que les ~20 dB vus plus haut sont un **signal reel** de qualite de separation, pas un mirage de la metrique.

Le controle de coherence va dans le meme sens : la somme des quatre stems recompose le melange a un residu de **-15 dB** pres. Les stems ne sont pas des inventions — ils recomposent bien le signal d'entree.

### Exercice 1 — Coherence d'energie d'un remix

Demucs re-separe en decoupant le signal en segments de 10 s qui se chevauchent, puis en recomposant les stems par moyenne ponderee. La somme des stems reproduit donc le melange — a un petit residu pres.

Ecrire `energy_balance(mix, stems)` qui retourne le niveau du residu en dB par rapport au melange :

    10 * log10( rms(mix - somme(stems)) / rms(mix) )

Un remix parfait donnerait -inf dB ; en pratique on attend mieux que -20 dB.

In [8]:
def energy_balance(mix, stems):
    """Niveau du residu du remix en dB par rapport au melange (exercice 1).

    mix   : tableau (n,) ou (n, ch)
    stems : dictionnaire {nom: tableau de meme forme que mix}
    Retour : float, le residu en dB (plus negatif = meilleur)
    """
    # TODO etudiant
    # Indice : sommez les valeurs de `stems`, soustrayez a `mix`,
    # puis comparez les RMS comme dans la cellule du temoin negatif.
    return None

### Exercice 2 — Difference entre deux architectures

`report` contient, pour chaque modele, le dictionnaire `sdr` par stem. Ecrire `model_delta(table_a, table_b)` qui retourne `{stem: sdr_a - sdr_b}` pour tous les stems presents dans les deux tables.

C'est la grandeur a lire en pratique : pas « quel modele est le meilleur en global », mais « sur quel stem le changement d'architecture rapporte le plus ».

In [9]:
def model_delta(table_a, table_b):
    """Dictionnaire {stem: sdr_a - sdr_b} (exercice 2).

    table_a, table_b : dictionnaires {stem: sdr} (ex. report[m]["sdr"])
    """
    # TODO etudiant
    # Indice : un comprehension sur table_a, en gardant les stems
    # presents dans table_b (cles communes).
    return None

### Exercice 3 — Le stem le plus difficile

Ecrire `worst_stem(report)` qui retourne le couple `(model, stem)` du SDR le plus bas de tout le benchmark, puis explique en une phrase ce que ce couple dit de la separation : qu'est-ce qui est dur a separer, et pour quel modele ?

In [10]:
def worst_stem(report):
    """Retourne (model, stem) du SDR minimal dans `report` (exercice 3).

    report : dictionnaire {model: {"sdr": {stem: sdr}, ...}}
    """
    # TODO etudiant
    # Indice : parcourir les modeles puis leurs stems ; garder le minimum vu.
    return None

## Exemple guide — mini-benchmark sur 3 pistes

Generaliser la mesure a plusieurs pistes : charger `n_tracks` pistes consecutives du split, les separer avec `benchmark_models` (2 modeles pour rester dans la VRAM), et dresser la table des **SDR medians** par stem et par modele.

Le median (et non la moyenne) est robuste a une piste atypique : c'est lui qu'on lira pour conclure a la meilleure architecture en general.

In [11]:
def charger_n_pistes(ds, n, max_pistes=20):
    """Charge n pistes completes consecutives du flux.

    Retour : liste de (nom, {instrument: (data, sr)})
    """
    pistes, par_piste = [], {}
    for row in ds:
        nom = row["path"].rsplit("/", 2)[1]
        par_piste.setdefault(nom, {})[row["instrument"]] = decoder_audio(row["audio"], clip_seconds)
        if len(par_piste[nom]) == 5:
            pistes.append((nom, par_piste.pop(nom)))
            if len(pistes) == n:
                return pistes
        if len(par_piste) + len(pistes) > max_pistes:
            break
    raise RuntimeError(f"seulement {len(pistes)} piste(s) complete(s) trouvee(s)")

from statistics import median

ds3 = load_dataset(dataset_id, split=split, streaming=True)
pistes = charger_n_pistes(ds3, n_tracks)
print("Pistes du mini-benchmark :")
for nom, _ in pistes:
    print(f"  - {nom}")

STEMS = ["vocals", "drums", "bass", "other"]
table = {m: {s: [] for s in STEMS} for m in benchmark_models}
for nom, sig in pistes:
    for model_name in benchmark_models:
        m = get_model(model_name).to(device).eval()
        with torch.no_grad():
            x = torch.from_numpy(sig["mixture"][0]).float().T.unsqueeze(0).to(device)
            out = apply_model(m, x, device=device, progress=False)
        for i, name in enumerate(m.sources):
            table[model_name][name].append(
                sdr_energie(sig[name][0], out[0, i].cpu().numpy().T))
        del m
        torch.cuda.empty_cache()

print(f"\nSDR median (dB) — {n_tracks} pistes, {len(benchmark_models)} modeles :")
print(f"{'modele':<15}" + "".join(f"{s:>10}" for s in STEMS))
for model_name in benchmark_models:
    print(f"{model_name:<15}" + "".join(
        f"{median(table[model_name][s]):10.1f}" for s in STEMS))
print(f"\nDuree totale du carnet : {time.time() - T_START:.0f} s")

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Pistes du mini-benchmark :
  - Alexander Ross - Velvet Curtain
  - Johnny Lokke - Whisper To A Scream
  - Hop Along - Sister Cities



SDR median (dB) — 3 pistes, 2 modeles :
modele             vocals     drums      bass     other
htdemucs             11.7       4.8       9.4      10.1
htdemucs_ft          12.0       5.3       9.8      10.7

Duree totale du carnet : 409 s


## Conclusion

Ce carnet a mesure, sur de vraies chansons avec leurs references studio, l'ecart entre trois generations d'architectures de separation. Les points a retenir :

- les nombres du benchmark sont des **mesures** : le temoin negatif (SDR effondre face a la reference d'une autre piste) et le controle de coherence d'energie l'etablissent ;
- la lecture utile est **par stem** (exercice 2), pas en global : chaque architecture a ses gains ;
- a architecture egale, **les donnees continuent de payer** (fine-tune) ;
- l'evaluation reste objective : la composante subjective (protocole MOS, proxies SNR/SI-SDR/LSD/STOI) est l'objet du carnet [04-19](04-19-Evaluation-MOS.ipynb).

## Pour aller plus loin

- [02-4 Demucs — le geste de base](../02-Advanced/02-4-Demucs-Source-Separation.ipynb)
- [04-19 Evaluation MOS — l'evaluation subjective](04-19-Evaluation-MOS.ipynb)
- MUSDB18 : Bertin et al., « MUSDB18: An Open Dataset and Benchmark for Music Source Separation », ISMIR 2016 — <https://arxiv.org/abs/1609.07229>
- EPIC #20568 (Origami Audio), Pli 2 — ce carnet en porte la case « separation avancee »